In [1]:
install.packages("sqldf")
install.packages("RSQLite")
install.packages("ggplot2")
install.packages("dplyr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [2]:
library(sqldf)
library(RSQLite)
library(ggplot2)
library(dplyr)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [7]:
unzip("/content/northstar_dataset (1).zip", exdir = getwd())
# Load all CSV files into R data frames
customers <- read.csv("northstar_dataset/customers.csv", stringsAsFactors = FALSE)
orders <- read.csv("northstar_dataset/orders.csv", stringsAsFactors = FALSE)
deliveries <- read.csv("northstar_dataset/deliveries.csv", stringsAsFactors = FALSE)
complaints <- read.csv("northstar_dataset/complaints.csv", stringsAsFactors = FALSE)
drivers <- read.csv("northstar_dataset/drivers.csv", stringsAsFactors = FALSE)
vehicles <- read.csv("northstar_dataset/vehicles.csv", stringsAsFactors = FALSE)
hubs <- read.csv("northstar_dataset/hubs.csv", stringsAsFactors = FALSE)
incidents <- read.csv("northstar_dataset/incidents.csv", stringsAsFactors = FALSE)
app_events <- read.csv("northstar_dataset/app_events.csv", stringsAsFactors = FALSE)

In [8]:
# Check that data loaded correctly
str(customers)
head(deliveries, 3)

'data.frame':	650 obs. of  9 variables:
 $ customer_id         : chr  "C0001" "C0002" "C0003" "C0004" ...
 $ age                 : int  26 61 66 75 26 41 54 70 34 23 ...
 $ home_zone           : chr  "North" "AIRPORT" "East" "CENTRAL" ...
 $ customer_type       : chr  "SME" "Consumer" "Consumer" "Consumer" ...
 $ signup_date         : chr  "2024-11-27 04:25:00" "2025-10-28 01:04:00" "2025-07-02 03:23:00" "2025-08-19 01:58:00" ...
 $ loyalty_score       : num  44.9 55.4 75.9 32.5 55.9 39.9 36.1 84.6 62.6 87.2 ...
 $ app_engagement_score: num  69.2 66.6 33.8 33 100 43.3 39 65.2 40.8 48.6 ...
 $ preferred_channel   : chr  "App" "App" "" "App" ...
 $ account_status      : chr  "Active" "Active" "Active" "Active" ...


,delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<int>,<int>,<dbl>,<dbl>
1,DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59.904311,Failed,17.26,1,0,3.07,12.05
2,DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00.000000,OnTime,10.34,1,0,5.00,13.41
3,DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32.366770,OnTime,7.92,0,0,4.98,8.51


In [9]:
#SQL query to calculate delivery performance by pickup zone
zone_performance <- sqldf("
  SELECT
    o.pickup_zone,
    COUNT(*) as total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) as delayed_count,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failed_count,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) / COUNT(*), 2) as delay_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 2) as failure_rate_pct
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.pickup_zone
  ORDER BY failure_rate_pct DESC
")

In [10]:
#Display results
print("=== Zone Performance Summary ===")
print(zone_performance)

[1] "=== Zone Performance Summary ==="
   pickup_zone total_deliveries delayed_count failed_count delay_rate_pct
1    RiverSide               66            12           14          18.18
2      Central               55            11           11          20.00
3      CENTRAL               55            16           11          29.09
4        North               37             6            7          16.22
5          Ctr               64            24           11          37.50
6        north               52             6            8          11.54
7        NORTH               46             9            7          19.57
8         EAST               78            14           11          17.95
9         West               51             8            7          15.69
10       South               83            15           10          18.07
11     Airport               67            18            8          26.87
12        WEST               63            13            7          20.63

In [11]:
#Additional insight: Overall company averages
overall_avg <- sqldf("
  SELECT
    ROUND(100.0 * SUM(CASE WHEN delivery_status = 'Delayed' THEN 1 ELSE 0 END) / COUNT(*), 2) as overall_delay_rate,
    ROUND(100.0 * SUM(CASE WHEN delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 2) as overall_failure_rate
  FROM deliveries
")
print("=== Company Overall Averages ===")
print(overall_avg)

[1] "=== Company Overall Averages ==="
  overall_delay_rate overall_failure_rate
1              21.26                13.89


In [12]:
 # SQL query to analyse driver training score vs delivery performance
driver_performance <- sqldf("
  SELECT
    CASE
      WHEN dr.training_score >= 80 THEN 'High (80-100)'
      WHEN dr.training_score >= 60 THEN 'Medium (60-79)'
      WHEN dr.training_score >= 40 THEN 'Low (40-59)'
      ELSE 'Very Low (<40)'
    END as training_level,
    COUNT(*) as total_deliveries,
    ROUND(AVG(dr.training_score), 1) as avg_training_score,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 2) as failure_rate,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) / COUNT(*), 2) as delay_rate,
    ROUND(AVG(d.customer_rating_post_delivery), 2) as avg_customer_rating
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  WHERE dr.training_score IS NOT NULL
  GROUP BY training_level
  ORDER BY avg_training_score DESC
")

print("=== Driver Training Level vs Delivery Performance ===")
print(driver_performance)

[1] "=== Driver Training Level vs Delivery Performance ==="
  training_level total_deliveries avg_training_score failure_rate delay_rate
1  High (80-100)              321               86.5        15.89      21.50
2 Medium (60-79)              491               71.1        13.85      21.79
3    Low (40-59)               98               53.1         9.18      21.43
  avg_customer_rating
1                3.86
2                3.86
3                3.92


In [13]:
# Query: Top 10 worst-performing drivers
worst_drivers <- sqldf("
  SELECT
    d.driver_id,
    dr.training_score,
    dr.driver_rating,
    COUNT(*) as total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) as failures,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 2) as failure_rate
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id
  HAVING total_deliveries >= 5
  ORDER BY failure_rate DESC
  LIMIT 10
")

print("=== Top 10 Worst-Performing Drivers ===")
print(worst_drivers)

[1] "=== Top 10 Worst-Performing Drivers ==="
   driver_id training_score driver_rating total_deliveries failures
1       D092           88.2          4.24                5        3
2       D104           87.7          3.45                7        4
3       D024           71.4          3.35                8        4
4       D010           70.0          3.95                7        3
5       D144           85.0          3.83                5        2
6       D143           68.5          4.14                5        2
7       D095           99.0          3.15                5        2
8       D005           69.7          4.14                5        2
9       D165           82.2          3.89                6        2
10      D133           88.2          3.99               12        4
   failure_rate
1         60.00
2         57.14
3         50.00
4         42.86
5         40.00
6         40.00
7         40.00
8         40.00
9         33.33
10        33.33


In [14]:
# SQL query to analyse complaints by hub
complaint_by_hub <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    h.zone as hub_zone,
    c.complaint_type,
    c.severity,
    COUNT(*) as complaint_count,
    ROUND(AVG(c.resolution_days), 1) as avg_resolution_days,
    ROUND(SUM(c.compensation_amount), 2) as total_compensation
  FROM complaints c
  JOIN orders o ON c.order_id = o.order_id
  JOIN deliveries d ON o.order_id = d.order_id
  JOIN hubs h ON d.hub_id = h.hub_id
  WHERE c.order_id IS NOT NULL
  GROUP BY h.hub_id, h.hub_name, c.complaint_type, c.severity
  ORDER BY complaint_count DESC
  LIMIT 20
")

print("=== Complaint Analysis by Hub ===")
print(complaint_by_hub)

[1] "=== Complaint Analysis by Hub ==="
   hub_id       hub_name  hub_zone    complaint_type severity complaint_count
1     H01 North Exchange     North             Delay   Medium               7
2     H03      East Dock      East      MissedPickup   Medium               7
3     H08  Midtown Relay   Central             Delay   Medium               7
4     H04      West Gate      West             Delay   Medium               6
5     H05   Central Core   Central             Delay   Medium               6
6     H07  Riverside Hub Riverside          AppIssue   Medium               6
7     H07  Riverside Hub Riverside      MissedPickup   Medium               6
8     H01 North Exchange     North             Delay      Low               5
9     H03      East Dock      East             Delay   Medium               5
10    H05   Central Core   Central          AppIssue     High               5
11    H07  Riverside Hub Riverside             Delay   Medium               5
12    H08  Midtown Relay

In [15]:
#Query: Hub performance summary
hub_summary <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    h.hub_type,
    COUNT(DISTINCT d.delivery_id) as total_deliveries,
    COUNT(DISTINCT c.complaint_id) as total_complaints,
    ROUND(100.0 * COUNT(DISTINCT c.complaint_id) / COUNT(DISTINCT d.delivery_id), 2) as complaint_rate_pct,
    ROUND(AVG(c.resolution_days), 1) as avg_resolution_days
  FROM hubs h
  LEFT JOIN deliveries d ON h.hub_id = d.hub_id
  LEFT JOIN orders o ON d.order_id = o.order_id
  LEFT JOIN complaints c ON o.order_id = c.order_id
  GROUP BY h.hub_id
  ORDER BY complaint_rate_pct DESC
")

print("=== Hub Performance Summary ===")
print(hub_summary)

[1] "=== Hub Performance Summary ==="
  hub_id       hub_name  hub_type total_deliveries total_complaints
1    H07  Riverside Hub Warehouse              115               33
2    H03      East Dock Warehouse              119               33
3    H08  Midtown Relay  Charging              128               35
4    H05   Central Core   Control              115               30
5    H01 North Exchange  Dispatch              136               32
6    H06    Airport Hub  Dispatch              104               23
7    H04      West Gate  Dispatch              127               28
8    H02     South Link  Dispatch              106               18
  complaint_rate_pct avg_resolution_days
1              28.70                 6.5
2              27.73                 7.9
3              27.34                 8.1
4              26.09                 9.4
5              23.53                 7.9
6              22.12                 9.1
7              22.05                 7.2
8              16.98  

In [16]:
#SQL query for financial analysis (cost vs revenue)
financial_analysis <- sqldf("
  SELECT
    o.service_type,
    o.pickup_zone,
    COUNT(*) as total_orders,
    ROUND(SUM(o.order_value), 2) as total_revenue,
    ROUND(SUM(d.fuel_or_charge_cost), 2) as total_fuel_cost,
    ROUND(SUM(o.order_value) - SUM(d.fuel_or_charge_cost), 2) as gross_profit,
    ROUND(100.0 * (SUM(o.order_value) - SUM(d.fuel_or_charge_cost)) / SUM(o.order_value), 2) as profit_margin_pct,
    ROUND(AVG(d.fuel_or_charge_cost / NULLIF(d.route_distance_km, 0)), 2) as cost_per_km
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  WHERE d.fuel_or_charge_cost IS NOT NULL AND o.order_value IS NOT NULL
  GROUP BY o.service_type, o.pickup_zone
  HAVING total_orders >= 5
  ORDER BY profit_margin_pct ASC
  LIMIT 15
")

print("=== Least Profitable Service Types by Zone ===")
print(financial_analysis)

[1] "=== Least Profitable Service Types by Zone ==="
   service_type pickup_zone total_orders total_revenue total_fuel_cost
1      Business     AIRPORT            5        333.06           91.12
2       Medical       north            7        402.43           99.16
3      Business        West            7        377.68           92.02
4       Medical        East            9        475.73          107.00
5      Business     Airport            8        748.78          152.98
6        Retail     Airport           14       1277.25          249.98
7        Parcel   RiverSide           17       1157.25          225.72
8     Passenger     Central           20       1234.97          238.04
9        Retail     AIRPORT           15       1409.88          269.17
10       Parcel     AIRPORT           11       1030.44          194.32
11    Passenger   RiverSide           19       1405.60          250.23
12       Parcel        EAST           23       1847.09          325.56
13     Business        E

In [17]:
#Query: Service type profitability comparison
service_profitability <- sqldf("
  SELECT
    o.service_type,
    COUNT(*) as total_orders,
    ROUND(AVG(o.order_value), 2) as avg_order_value,
    ROUND(AVG(d.fuel_or_charge_cost), 2) as avg_fuel_cost,
    ROUND(AVG(o.order_value) - AVG(d.fuel_or_charge_cost), 2) as avg_profit,
    ROUND(100.0 * (AVG(o.order_value) - AVG(d.fuel_or_charge_cost)) / AVG(o.order_value), 2) as avg_margin_pct
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  WHERE d.fuel_or_charge_cost IS NOT NULL
  GROUP BY o.service_type
  ORDER BY avg_margin_pct ASC
")

print("=== Profitability by Service Type ===")
print(service_profitability)

[1] "=== Profitability by Service Type ==="
  service_type total_orders avg_order_value avg_fuel_cost avg_profit
1       Retail          224           86.81         12.97      73.83
2      Medical          108           86.53         12.77      73.75
3       Parcel          230           90.15         13.08      77.07
4     Business          126           97.45         13.14      84.31
5    Passenger          262           97.19         12.40      84.79
  avg_margin_pct
1          85.05
2          85.24
3          85.49
4          86.51
5          87.24


In [18]:
#Query: Identify loss-making deliveries
loss_making <- sqldf("
  SELECT
    d.delivery_id,
    o.service_type,
    o.order_value,
    d.fuel_or_charge_cost,
    o.order_value - d.fuel_or_charge_cost as profit,
    d.delivery_status,
    d.route_distance_km,
    d.manual_route_override_count
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  WHERE d.fuel_or_charge_cost > o.order_value
  ORDER BY profit ASC
  LIMIT 10
")

print("=== Loss-Making Deliveries (Cost > Revenue) ===")
print(loss_making)

[1] "=== Loss-Making Deliveries (Cost > Revenue) ==="
   delivery_id service_type order_value fuel_or_charge_cost profit
1      DL00739     Business        6.28               15.70  -9.42
2      DL00897       Retail       20.28               29.43  -9.15
3      DL00422      Medical        2.04               10.37  -8.33
4      DL00081    Passenger        5.92               12.68  -6.76
5      DL00879       Retail        4.22               10.97  -6.75
6      DL00297     Business       16.12               22.27  -6.15
7      DL00157       Retail        7.74               13.52  -5.78
8      DL00933       Parcel        8.61               14.09  -5.48
9      DL00545       Retail        8.17               13.31  -5.14
10     DL00673       Parcel        7.14               11.92  -4.78
   delivery_status route_distance_km manual_route_override_count
1          Delayed              7.60                           1
2           OnTime             33.64                           2
3           On

In [19]:
#Create a summary table of key findings
findings_summary <- data.frame(
  Finding = c(
    "Highest failure rate zone",
    "Highest delay rate zone",
    "Driver training impact",
    "Highest complaint hub",
    "Least profitable service",
    "Most profitable service"
  ),
  Value = c(
    zone_performance$pickup_zone[which.max(zone_performance$failure_rate_pct)],
    zone_performance$pickup_zone[which.max(zone_performance$delay_rate_pct)],
    paste0("Drivers with <60 training score have ",
           driver_performance$failure_rate[driver_performance$training_level == "Low (40-59)"],
           "% failure rate"),
    hub_summary$hub_name[which.max(hub_summary$complaint_rate_pct)],
    service_profitability$service_type[which.min(service_profitability$avg_margin_pct)],
    service_profitability$service_type[which.max(service_profitability$avg_margin_pct)]
  )
)

print("=== KEY FINDINGS SUMMARY ===")
print(findings_summary)

[1] "=== KEY FINDINGS SUMMARY ==="
                    Finding
1 Highest failure rate zone
2   Highest delay rate zone
3    Driver training impact
4     Highest complaint hub
5  Least profitable service
6   Most profitable service
                                                    Value
1                                               RiverSide
2                                                     Ctr
3 Drivers with <60 training score have 9.18% failure rate
4                                           Riverside Hub
5                                                  Retail
6                                               Passenger


In [20]:
#Save results to CSV for reference
write.csv(zone_performance, "zone_performance_analysis.csv", row.names = FALSE)
write.csv(driver_performance, "driver_performance_analysis.csv", row.names = FALSE)
write.csv(financial_analysis, "financial_analysis.csv", row.names = FALSE)

print("Analysis complete. Results saved to CSV files.")

[1] "Analysis complete. Results saved to CSV files."


In [ ]:
unzip("/content/northstar_dataset (1).zip", exdir = getwd())

In [5]:
list.files()

[1] "northstar_dataset (1).zip" "sample_data"

In [6]:
list.files('northstar_dataset')

character(0)